<div align="center">

<a href="https://colab.research.google.com/github/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-04/NB04_fuzzy_control.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 04: Fuzzy Logic and Fuzzy Control

**Artificial Intelligence Applications in Engineering (MUH-920 Mühendislikte Yapay Zeka Uygulamaları)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)<br>[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. Last update: September 2026.</sub>

</div>

## How to use this notebook

This notebook is a self-contained version of the week: The lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. No earlier Python experience is assumed: Each new piece of Python appears in a cell marked **Python move**, which explains the syntax right where it is first needed. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: Open them only after a genuine attempt.

The animations of the lecture run on the [lecture page](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-04/lecture.html), and the interactive lab with the self-assessment is available at <https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-04/lab.html>. Cells that download public data need an internet connection. When a source is unreachable, the notebook falls back to a documented synthetic stand-in so that every later cell still runs.

## Overview

Operators describe processes in words: The level is a bit low and falling slowly, so open the valve a little. Fuzzy logic turns such statements into computation. Zadeh introduced fuzzy sets, whose members belong to a degree between zero and one [1], and linguistic variables that take words as values [2]. Mamdani and Assilian used them to control a steam engine with rules taken from operators [3], and within a decade fuzzy controllers ran cement kilns and a subway line [4, 5]. This week builds a fuzzy inference system from scratch, uses it to control the level of a tank, compares Mamdani and Takagi-Sugeno models, and shows how adaptive neuro-fuzzy systems learn their parameters from data [6, 7].

**Estimated study time:** 9 to 11 hours.

## Learning outcomes

By the end of the week, students are expected to define fuzzy sets with triangular, trapezoidal and Gaussian membership functions, to apply fuzzy complement, intersection and union, to carry out Mamdani inference by hand from fuzzification to centroid defuzzification, to design a rule table for a two-input controller and interpret its control surface, to distinguish Mamdani from Takagi-Sugeno models, and to implement and simulate a fuzzy controller in Python, both from scratch and with scikit-fuzzy.

## Python in this week

The lecture ends with Python step 4: NumPy arrays and first plots. It covers NumPy arrays, element-wise arithmetic, functions on arrays, boolean masks and Matplotlib plots, applied to membership functions and defuzzification [8, 9]. The hands-on part of the notebook opens with the same step as runnable cells and short exercises with immediate feedback, and its later sections mark further constructs as Python moves.

## Vagueness and fuzzy sets

Week 3 ended with a weakness of crisp rules: A threshold of 70 degrees makes a reading of 69.9 normal and a reading of 70.1 overheated. Engineers rarely think in such steps. A temperature is more or less high, a load is fairly heavy, a surface is quite rough. Zadeh proposed fuzzy sets to represent such graded concepts [1]. A fuzzy set A on a universe of discourse X is defined by a membership function that assigns every element x a degree between 0 and 1. A room at 23 degrees may belong to the set of warm rooms with degree 0.6 and to the set of comfortable rooms with degree 0.4.

Vagueness is not probability. A membership degree of 0.6 does not say that the room is warm with probability 0.6. It says that 23 degrees fits the concept "warm" to degree 0.6, and there is nothing random about it. Probability describes uncertainty about whether an event happens; fuzzy membership describes how well a precisely known value fits an imprecise concept. The two can be combined, but they answer different questions.

Common membership functions are triangular, defined by a left foot, a peak and a right foot; trapezoidal, with a flat top; and Gaussian, with a centre and a width. The support of a fuzzy set is where its membership is positive, the core is where it equals one, and an alpha-cut is the crisp set of elements whose membership is at least alpha. A linguistic variable, such as tank level, takes linguistic values, such as low, okay and high, each represented by a fuzzy set on the same universe [2].


![A crisp set and fuzzy sets for the concept "warm room". The fuzzy sets overlap, so a temperature can belong partly to two neighbouring concepts [1].](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-04/figures/w04_fig1.png)

*Figure 4.1. A crisp set and fuzzy sets for the concept "warm room". The fuzzy sets overlap, so a temperature can belong partly to two neighbouring concepts [1].*



<details>
<summary><b>Check your understanding.</b> A water temperature of 58 degrees has membership 0.7 in the fuzzy set &quot;hot&quot;. What does this mean?</summary>

A. The water is hot with probability 0.7  
B. 58 degrees fits the concept "hot" to degree 0.7  
C. 70 percent of the water is hot  
D. The measurement has an error of 0.7 degrees

**Answer: B.** Membership measures compatibility with a vague concept, not the probability of a random event.

</details>

## Operations on fuzzy sets

The classical set operations generalise. The complement NOT A has membership 1 minus the membership of A. The intersection A AND B is usually computed with the minimum of the two memberships, and the union A OR B with the maximum [1]. Other choices are possible: The product is also a valid intersection, called a t-norm, and the probabilistic sum a + b - ab is a valid union, called an s-norm. The choice changes the smoothness of the resulting controller but not the principle. Some classical laws fail on purpose: A AND NOT A need not be empty, because a temperature can be somewhat warm and somewhat not warm at the same time.

## Mamdani inference

A fuzzy rule has linguistic conditions and a linguistic conclusion: IF level error is low AND level rate is none THEN valve change is open fast. Mamdani inference evaluates a rule base in five steps [3, 10]. First, fuzzification computes the membership of each crisp input in each of its fuzzy sets. Second, each rule receives a firing strength, the minimum of the memberships in its conditions. Third, implication clips the output fuzzy set of each rule at its firing strength. Fourth, aggregation combines the clipped sets of all rules with the maximum. Fifth, defuzzification turns the aggregated fuzzy set into one crisp output, most often with the centroid, the centre of gravity of the area under the aggregated membership function.


> **Animation: Inside a Mamdani controller.** Move the two inputs of a tank level controller. The bars show fuzzification, the clipped shapes show the rules that fire, and the vertical line marks the centroid that becomes the valve command. Run it in the [web version of the lecture](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-04/lecture.html#anim-mamdani).


![Control surface of the five-rule tank controller of the notebook. The fuzzy controller is a smooth nonlinear map from level error and level rate to valve change.](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-04/figures/w04_fig2.png)

*Figure 4.2. Control surface of the five-rule tank controller of the notebook. The fuzzy controller is a smooth nonlinear map from level error and level rate to valve change.*



The controller defined by a rule base is a nonlinear function from inputs to output, and its control surface can be drawn for two inputs. Overlapping membership functions make the surface smooth; more sets and rules add detail. A rule table, with one input along the rows and the other along the columns, is the usual design tool: Every cell holds the conclusion for one combination of linguistic values, and gaps in the table leave parts of the input space without any action. Symmetry of the table usually leads to symmetric behaviour of the loop.


<details>
<summary><b>Check your understanding.</b> Two rules fire with strengths 0.3 and 0.8 and conclude &quot;open slow&quot; and &quot;open fast&quot;. What happens in Mamdani inference before defuzzification?</summary>

A. Only the stronger rule is kept  
B. Each output set is clipped at its rule strength and the clipped sets are combined with the maximum  
C. The strengths are added to 1.1 and the output is saturated  
D. The inputs are fuzzified again with new membership functions

**Answer: B.** Implication clips each consequent at its firing strength and aggregation takes the pointwise maximum of the clipped sets.

</details>

## Takagi-Sugeno models and learning fuzzy systems

Takagi and Sugeno replaced the fuzzy conclusion by a function of the inputs: IF x is A AND y is B THEN z = p x + q y + r [6]. The output is the average of the rule outputs weighted by their firing strengths, so no defuzzification of a shape is needed. A zero-order Sugeno model has constant conclusions and behaves much like a Mamdani model with narrow output sets; a first-order model blends local linear models, which makes it attractive for modelling and for gain scheduling of controllers.

Because the Sugeno output is a differentiable function of its parameters, the parameters can be learned from data. ANFIS represents a Sugeno system as a five-layer adaptive network and trains the premise parameters with gradient descent and the linear consequent parameters with least squares [7]. Metaheuristic training is an alternative when gradients are unreliable: In one study, ANFIS supported by the vortex optimisation algorithm forecast chaotic electroencephalogram time series [11]. Such neuro-fuzzy systems link this week to the optimisation of Weeks 5 and 6 and to the neural networks of Week 10.


<details>
<summary><b>Check your understanding.</b> What distinguishes a first-order Takagi-Sugeno rule from a Mamdani rule?</summary>

A. Its conditions are crisp  
B. Its conclusion is a linear function of the inputs instead of a fuzzy set  
C. It cannot be combined with other rules  
D. It uses the maximum instead of the minimum for AND

**Answer: B.** The consequent z = p x + q y + r is a local linear model; the rule outputs are averaged with the firing strengths as weights.

</details>

## Fuzzy control in engineering

The first industrial success was the control of a cement kiln in Denmark, where operators' rules for burning zone temperature and oxygen were encoded as fuzzy rules [4]. The Sendai subway in Japan used predictive fuzzy control for automatic train operation, which improved riding comfort and stopping accuracy [5]. Fuzzy controllers later appeared in washing machines, cameras, heating systems, water treatment dosing and vehicle subsystems, and textbooks collect applications across civil, mechanical, chemical and electrical engineering [12].

Fuzzy control is most useful when a good mathematical model is missing but operator experience exists, when the process is nonlinear, and when smooth, explainable behaviour matters. It does not remove the need for analysis: Stability, robustness to disturbances and the effect of measurement noise must still be checked, for example by simulation over the operating range, as in the notebook. When a reliable model exists, classical or model-based control is often the better engineering choice.

# Hands-on lab

The notebook implements membership functions, fuzzy operations and Mamdani inference with NumPy, shows every inference step for one input, draws the control surface, and simulates the controller on a nonlinear tank model against an on-off controller. It then rebuilds the same controller with scikit-fuzzy [13], compares it with a zero-order Sugeno version, and closes with exercises on membership functions, centroids and rule design.

## 0. Setup

**Python move: installing a library.** Colab has NumPy and Matplotlib but not scikit-fuzzy. A line starting with `!` runs a shell command, so `!pip install scikit-fuzzy` installs it. The `try`/`except ImportError` pattern installs it only when needed.

In [ ]:
try:
    import skfuzzy
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-fuzzy"], check=False)
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] {'Correct.' if ok else 'Not yet. Check the logic and run again.'}")

## Python step 4: NumPy arrays and first plots

This part follows the Python step at the end of the lecture. Run the cells in order, compare each result with the explanation, and change values to see what happens. The exercises at the end of the step give immediate feedback.

### From lists to arrays

Engineering calculations usually apply one formula to many values. NumPy provides the array, a grid of numbers of one type on which arithmetic works element by element [8]. Multiplying an array by 10 multiplies every element, whereas multiplying a list by 10 would repeat the list ten times. Arrays have a `shape`, the number of elements along each axis, and a `dtype`, the type of their elements. `np.linspace(a, b, n)` creates n evenly spaced values from a to b, which is the usual way to sample a variable such as a tank level.

In [ ]:
import numpy as np

levels_cm = np.array([12.0, 35.5, 50.0, 71.2, 94.0])
print(levels_cm * 10)                 # millimetres, element by element
print(levels_cm.shape, levels_cm.dtype, levels_cm.mean())
x = np.linspace(0, 100, 11)           # 11 evenly spaced points from 0 to 100
print(x)

### Functions that work on arrays

A function written with array operations accepts a single number as well as a whole array. The triangular membership function of fuzzy logic rises linearly from a foot a to a peak b and falls to a second foot c. `np.minimum` and `np.maximum` compare element by element, which expresses the whole shape in one line.

In [ ]:
def tri(x, a, b, c):
    """Triangular membership function with feet a and c and peak b."""
    return np.maximum(np.minimum((x - a) / (b - a), (c - x) / (c - b)), 0.0)


print(tri(35.0, 0, 25, 50))
print(np.round(tri(x, 0, 25, 50), 2))

### Aggregation and boolean masks

Functions such as `np.sum`, `np.max` and `np.argmax` reduce an array to a single value or position. Comparisons produce boolean arrays, called masks, which count values when summed and select values when used as an index. The centroid of a fuzzy set, the defuzzification method of the Mamdani controller, is a weighted mean: The sum of x times membership is divided by the sum of membership. The example clips the output set "open" of a valve at a rule strength of 0.6.

In [ ]:
valve = np.linspace(0, 100, 201)          # valve opening in percent
open_set = tri(valve, 40, 70, 100)        # output set "open"
clipped = np.minimum(open_set, 0.6)       # a rule with strength 0.6 clips the set
centroid = np.sum(valve * clipped) / np.sum(clipped)
print(f"centroid: {centroid:.1f} % opening")
print("points above 0.5:", np.sum(clipped > 0.5))
print("openings with membership above 0.5:", valve[clipped > 0.5][[0, -1]])

The last line uses the mask as an index to keep only the matching openings, and then selects the first and the last of them.

### Plots

Matplotlib draws figures from arrays [9]. `plt.plot(x, y)` connects points with lines, the `label` argument names each curve for the legend, and axis labels with units belong on every engineering plot. The code below draws the three level sets of the tank controller; the notebook shows the figure.

In [ ]:
import matplotlib.pyplot as plt

level = np.linspace(0, 100, 201)
plt.figure(figsize=(6, 2.8))
plt.plot(level, tri(level, -50, 0, 50), label="low")
plt.plot(level, tri(level, 0, 50, 100), label="medium")
plt.plot(level, tri(level, 50, 100, 150), label="high")
plt.xlabel("tank level (cm)")
plt.ylabel("membership")
plt.legend()
plt.show()

Feet placed outside the range, such as -50 and 150, make the outer sets start or end at full membership at the edges of the range, which is how a controller treats a tank that is empty or full.

**Quick check.** After a = np.array([1.0, 2.0, 3.0]), what is a * 2 + 1?

A. array([3., 5., 7.])  
B. [1.0, 2.0, 3.0, 1.0, 2.0, 3.0, 1]  
C. An error  
D. 15.0

<details><summary>Show the answer</summary>

**A.** Arithmetic on arrays acts element by element. With a list, * 2 would repeat the list, and adding 1 would raise an error.

</details>

### Exercises for Python step 4

**Exercise 4.1.** Compute the membership of a level of 62 cm in the set "medium", `tri(level, 0, 50, 100)`, and store it in `py_mu_medium`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py4_mu_medium_reference():
    return min((62 - 0) / 50, (100 - 62) / 50)

In [ ]:
py_mu_medium = None
check("Exercise 4.1", py_mu_medium, _py4_mu_medium_reference())

**Exercise 4.2.** The output set `tri(valve, 40, 60, 100)` is clipped at 0.5. Compute the centroid of the clipped set and store it in `py_centroid`. Why does it lie to the right of 60?

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py4_centroid_reference():
    import numpy as np
    v = np.linspace(0, 100, 201)
    s = np.minimum(np.maximum(np.minimum((v - 40) / 20, (100 - v) / 40), 0.0), 0.5)
    return float(np.sum(v * s) / np.sum(s))

In [ ]:
py_centroid = None
check("Exercise 4.2", py_centroid, _py4_centroid_reference())

**Exercise 4.3.** Use a mask to find the share of the 201 points in `level` at which the membership of "high" exceeds 0.2, and store the share, a number between 0 and 1, in `py_share_high`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py4_share_high_reference():
    import numpy as np
    lv = np.linspace(0, 100, 201)
    hi = np.maximum(np.minimum((lv - 50) / 50, (150 - lv) / 50), 0.0)
    return float(np.mean(hi > 0.2))

In [ ]:
py_share_high = None
check("Exercise 4.3", py_share_high, _py4_share_high_reference())

## 1. Membership functions with NumPy

**Python move: vectorised functions.** `trimf` works on a single number and on a whole array, because NumPy applies `np.minimum` and `np.maximum` element by element [8].

In [ ]:
def trimf(x, a, b, c):
    x = np.asarray(x, dtype=float)
    left = np.where(b > a, (x - a) / (b - a + 1e-12), 1.0)
    right = np.where(c > b, (c - x) / (c - b + 1e-12), 1.0)
    return np.clip(np.minimum(left, right), 0, 1)


def gaussmf(x, mean, sigma):
    return np.exp(-0.5 * ((np.asarray(x, dtype=float) - mean) / sigma) ** 2)


t = np.linspace(0, 40, 401)
cool, comfortable, warm = trimf(t, 5, 12, 20), trimf(t, 16, 21, 26), trimf(t, 22, 30, 38)
plt.figure(figsize=(7, 2.8))
for mu, name in [(cool, "cool"), (comfortable, "comfortable"), (warm, "warm")]:
    plt.plot(t, mu, label=name)
plt.plot(t, gaussmf(t, 21, 2.5), ls="--", label="comfortable, Gaussian")
plt.xlabel("room temperature (C)"); plt.ylabel("membership"); plt.legend(fontsize=8); plt.show()
print("23 C is comfortable to degree", float(trimf(23, 16, 21, 26)), "and warm to degree", float(trimf(23, 22, 30, 38)))

### Your turn, exercise 1

Write `trapmf(x, a, b, c, d)` for a trapezoid that rises from a to b, stays at 1 from b to c and falls from c to d. Evaluate it at x = 11 for (a, b, c, d) = (2, 5, 9, 14) and store the value in `mu_trap`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _trapmf_impl(x, a, b, c, d):
    x = np.asarray(x, dtype=float)
    return np.clip(np.minimum(np.minimum((x - a) / (b - a), 1.0), (d - x) / (d - c)), 0, 1)


def _mu_trap_reference():
    return float(_trapmf_impl(11, 2, 5, 9, 14))

In [ ]:
mu_trap = None   # your computation
check("Exercise 1", mu_trap, _mu_trap_reference())

## 2. A tank level controller, step by step

Inputs: level error e = set point minus level (m) and level rate r (m/s). Output: rate of change of the valve opening (1/s). Five rules follow operator logic [3, 10]:

1. IF error is low THEN open fast.
2. IF error is high THEN close fast.
3. IF error is okay AND rate is none THEN no change.
4. IF error is okay AND rate is positive THEN close slow.
5. IF error is okay AND rate is negative THEN open slow.

In [ ]:
E_SETS = {"high": (-1, -1, 0), "okay": (-0.4, 0, 0.4), "low": (0, 1, 1)}
R_SETS = {"negative": (-0.1, -0.1, 0), "none": (-0.03, 0, 0.03), "positive": (0, 0.1, 0.1)}
O_SETS = {"close_fast": (-0.07, -0.05, -0.03), "close_slow": (-0.035, -0.02, -0.005), "no_change": (-0.012, 0, 0.012),
          "open_slow": (0.005, 0.02, 0.035), "open_fast": (0.03, 0.05, 0.07)}
RULES = [("low", None, "open_fast"), ("high", None, "close_fast"), ("okay", "none", "no_change"),
         ("okay", "positive", "close_slow"), ("okay", "negative", "open_slow")]
Y = np.linspace(-0.07, 0.07, 561)          # output universe


def mamdani(e, r, rules=RULES, show=False):
    e, r = float(np.clip(e, -1, 1)), float(np.clip(r, -0.1, 0.1))
    aggregated = np.zeros_like(Y)
    strengths = []
    for e_set, r_set, out in rules:
        s = float(trimf(e, *E_SETS[e_set]))
        if r_set is not None:
            s = min(s, float(trimf(r, *R_SETS[r_set])))              # AND as minimum
        strengths.append(s)
        clipped = np.minimum(s, trimf(Y, *O_SETS[out]))              # implication: clip
        aggregated = np.maximum(aggregated, clipped)                 # aggregation: max
    crisp = float(np.sum(Y * aggregated) / np.sum(aggregated)) if aggregated.sum() > 0 else 0.0
    if show:
        print("firing strengths:", [round(s, 3) for s in strengths])
        plt.figure(figsize=(6, 2.6))
        for name, p in O_SETS.items():
            plt.plot(Y, trimf(Y, *p), color="gray", lw=0.8, ls="--")
        plt.fill_between(Y, aggregated, alpha=0.5, label="aggregated")
        plt.axvline(crisp, color="C3", lw=2, label=f"centroid {crisp:.4f}")
        plt.xlabel("valve change (1/s)"); plt.legend(fontsize=8); plt.show()
    return crisp


mamdani(0.25, 0.02, show=True)

## 3. The control surface

**Python move: np.meshgrid.** `np.meshgrid` builds all combinations of two input grids, so the controller can be evaluated on a table of inputs and drawn as a surface.

In [ ]:
ee, rr = np.meshgrid(np.linspace(-1, 1, 41), np.linspace(-0.1, 0.1, 41))
zz = np.vectorize(mamdani)(ee, rr)
plt.figure(figsize=(5.2, 4))
plt.contourf(ee, rr, zz, levels=21, cmap="RdYlGn"); plt.colorbar(label="valve change (1/s)")
plt.xlabel("level error (m)"); plt.ylabel("level rate (m/s)"); plt.title("Mamdani control surface"); plt.show()

## 4. Closed-loop simulation

The tank has area A = 2 m2 and drains through an outlet with flow 0.6 sqrt(h). The valve delivers up to 1 m3/s. The set point steps from 1.0 m to 1.5 m and then to 0.8 m. The fuzzy controller changes the valve gradually; an on-off controller opens fully or closes fully.

In [ ]:
def simulate(controller, steps=1200, dt=0.5):
    h, v, prev = 1.0, 0.6, 1.0
    levels, set_points, valves = [], [], []
    for k in range(steps):
        sp = 1.0 if k < 300 else (1.5 if k < 700 else 0.8)
        e, r = sp - h, (h - prev) / dt
        prev = h
        v = controller(e, r, v, dt)
        h = max(0.0, h + (1.0 * v - 0.6 * np.sqrt(h)) * dt / 2.0)
        levels.append(h); set_points.append(sp); valves.append(v)
    return np.array(levels), np.array(set_points), np.array(valves)


fuzzy_ctrl = lambda e, r, v, dt: float(np.clip(v + mamdani(e, r) * dt, 0, 1))
on_off = lambda e, r, v, dt: 1.0 if e > 0 else 0.0
results = {"fuzzy (Mamdani)": simulate(fuzzy_ctrl), "on-off": simulate(on_off)}
time = np.arange(1200) * 0.5
fig, ax = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True)
for name, (L, SP, V) in results.items():
    ax[0].plot(time, L, label=name); ax[1].plot(time, V, label=name)
    print(f"{name:16s} mean |error| {np.mean(np.abs(SP - L)):.3f} m")
ax[0].plot(time, SP, "k--", lw=0.8, label="set point"); ax[0].set_ylabel("level (m)"); ax[0].legend(fontsize=8)
ax[1].set_ylabel("valve opening"); ax[1].set_xlabel("time (s)"); plt.show()

## 5. The same controller with scikit-fuzzy

scikit-fuzzy describes antecedents, consequents and rules as objects [13]. Its default defuzzification is the centroid, so the result should match the from-scratch version closely.

In [ ]:
import skfuzzy as fuzz
from skfuzzy import control as ctrl

error = ctrl.Antecedent(np.linspace(-1, 1, 401), "error")
rate = ctrl.Antecedent(np.linspace(-0.1, 0.1, 401), "rate")
valve = ctrl.Consequent(Y, "valve")
for name, p in E_SETS.items():
    error[name] = fuzz.trimf(error.universe, list(p))
for name, p in R_SETS.items():
    rate[name] = fuzz.trimf(rate.universe, list(p))
for name, p in O_SETS.items():
    valve[name] = fuzz.trimf(valve.universe, list(p))
rules = [ctrl.Rule(error["low"], valve["open_fast"]), ctrl.Rule(error["high"], valve["close_fast"]),
         ctrl.Rule(error["okay"] & rate["none"], valve["no_change"]), ctrl.Rule(error["okay"] & rate["positive"], valve["close_slow"]),
         ctrl.Rule(error["okay"] & rate["negative"], valve["open_slow"])]
sim = ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules))
sim.input["error"], sim.input["rate"] = 0.25, 0.02
sim.compute()
print(f"scikit-fuzzy: {sim.output['valve']:.4f}   from scratch: {mamdani(0.25, 0.02):.4f}")

## 6. A zero-order Sugeno version

In a zero-order Takagi-Sugeno model, each rule outputs a constant and the result is the average of these constants weighted by the firing strengths [6]. No output shape needs to be defuzzified.

In [ ]:
CENTRES = {name: p[1] for name, p in O_SETS.items()}


def sugeno(e, r):
    e, r = float(np.clip(e, -1, 1)), float(np.clip(r, -0.1, 0.1))
    w, z = [], []
    for e_set, r_set, out in RULES:
        s = float(trimf(e, *E_SETS[e_set]))
        if r_set is not None:
            s = min(s, float(trimf(r, *R_SETS[r_set])))
        w.append(s); z.append(CENTRES[out])
    return float(np.dot(w, z) / np.sum(w)) if np.sum(w) > 0 else 0.0


L_s, SP_s, _ = simulate(lambda e, r, v, dt: float(np.clip(v + sugeno(e, r) * dt, 0, 1)))
print(f"Sugeno mean |error| {np.mean(np.abs(SP_s - L_s)):.3f} m; output at (0.25, 0.02): {sugeno(0.25, 0.02):.4f}")

### Your turn, exercise 2

Compute the Mamdani output by hand-written NumPy for e = -0.3 and r = 0.05 with the rules above, without calling `mamdani`: fuzzify, compute the five strengths, clip, aggregate and take the centroid on `Y`. Store the crisp output in `u_manual`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _u_manual_reference():
    return mamdani(-0.3, 0.05)

In [ ]:
u_manual = None   # your computation
check("Exercise 2", u_manual, _u_manual_reference(), tol=1e-4)

### Your turn, exercise 3

Make the "okay" error set narrower, (-0.2, 0, 0.2), rerun the closed-loop simulation with the Mamdani controller and store the new mean absolute error in `mae_narrow`. Does a narrower set make the controller more aggressive or more gentle near the set point?

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _mae_narrow_reference():
    global E_SETS
    saved = dict(E_SETS)
    E_SETS = dict(E_SETS, okay=(-0.2, 0, 0.2))
    L, SP, _ = simulate(fuzzy_ctrl)
    E_SETS = saved
    return float(np.mean(np.abs(SP - L)))

In [ ]:
mae_narrow = None   # your computation
check("Exercise 3", mae_narrow, _mae_narrow_reference(), tol=1e-3)

## Discipline challenges

Fuzzy logic suits quantities that experts judge with words. Choose a row, define linguistic variables with membership functions, write a rule table, and test the result on a few cases.

| Department | Challenge |
|---|---|
| Environmental Engineering | Coagulant dosing from raw-water turbidity and pH, or an air quality index from several pollutants. |
| Food Engineering | Drying control from product moisture and air temperature, or sensory quality grading from colour and texture scores. |
| Civil Engineering | Condition rating of a bridge element from crack width, corrosion and deflection. |
| Mining Engineering | Ground stability assessment from joint spacing, water inflow and blasting vibration. |
| Electrical and Electronics Engineering | Battery charging current from temperature and state of charge. |
| Mechanical Engineering and Automotive Engineering | Wheel slip control or engine cooling fan control from temperature and its rate. |
| Chemical Engineering | Reactor temperature control with a two-input fuzzy PI structure. |
| Industrial Engineering | Supplier selection with fuzzy scores for price, quality and delivery. |
| Textile Engineering | Fabric hand evaluation from bending, friction and compression measurements. |
| Earth Sciences Engineering and Geophysical Engineering | Landslide susceptibility from slope, lithology and rainfall classes. |
| Mathematics and Statistics | Compare t-norms and s-norms and prove which classical set laws fail for min and max. |

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-04/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which quantity in your field is naturally described with vague words by practitioners? Sketch three membership functions for it.
2. The tank controller in the notebook has five rules. What would you change first if the level overshot the set point, and why?
3. How did vectorised NumPy code change the way you wrote the membership functions compared with the loops of Week 1?

## Weekly task and submission

Design a fuzzy controller or a fuzzy assessment model for a problem of your department with at least two inputs, three sets per input and a complete rule table. Implement it with the from-scratch code or with scikit-fuzzy, plot its control surface, and test it on a simple simulation or on at least ten cases. Report the design choices and the behaviour in about 500 words and cite at least three works from this week's references [3, 10, 12].

The weekly task is optional and supports self-learning and a personal portfolio. During an active semester, it can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com. Optional work carries no separate weight, but the instructor may take it into account as a discretionary adjustment of the midterm component of the grade.

## Research and report assignment (optional)

**Fuzzy systems in engineering decision making.** Review fuzzy control or fuzzy decision support in one domain, such as water treatment, building energy, traffic, mining safety or food quality. Compare Mamdani, Takagi-Sugeno and adaptive neuro-fuzzy approaches and discuss how the membership functions and rules were obtained, from experts, from data or by optimisation [6, 7, 11].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com. It carries no separate weight, but the instructor may take it into account as a discretionary adjustment of the midterm component of the grade. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and uses the template in `exams/REPORT_TEMPLATE.md`.

## References

[1] Zadeh, L. A. (1965). Fuzzy sets. *Information and Control*, *8*(3), 338-353. <https://doi.org/10.1016/S0019-9958(65)90241-X>

[2] Zadeh, L. A. (1973). Outline of a new approach to the analysis of complex systems and decision processes. *IEEE Transactions on Systems, Man, and Cybernetics*, *SMC-3*(1), 28-44. <https://doi.org/10.1109/TSMC.1973.5408575>

[3] Mamdani, E. H., & Assilian, S. (1975). An experiment in linguistic synthesis with a fuzzy logic controller. *International Journal of Man-Machine Studies*, *7*(1), 1-13. <https://doi.org/10.1016/S0020-7373(75)80002-2>

[4] Holmblad, L. P., & Østergaard, J.-J. (1982). Control of a cement kiln by fuzzy logic. In *Fuzzy Information and Decision Processes (M. M. Gupta & E. Sanchez, Eds.)* (pp. 389-399). North-Holland.

[5] Yasunobu, S., & Miyamoto, S. (1985). Automatic train operation system by predictive fuzzy control. In *Industrial Applications of Fuzzy Control (M. Sugeno, Ed.)* (pp. 1-18). North-Holland.

[6] Takagi, T., & Sugeno, M. (1985). Fuzzy identification of systems and its applications to modeling and control. *IEEE Transactions on Systems, Man, and Cybernetics*, *SMC-15*(1), 116-132. <https://doi.org/10.1109/TSMC.1985.6313399>

[7] Jang, J.-S. R. (1993). ANFIS: Adaptive-network-based fuzzy inference system. *IEEE Transactions on Systems, Man, and Cybernetics*, *23*(3), 665-685. <https://doi.org/10.1109/21.256541>

[8] Harris, C. R., Millman, K. J., van der Walt, S. J., et al. (2020). Array programming with NumPy. *Nature*, *585*(7825), 357-362. <https://doi.org/10.1038/s41586-020-2649-2>

[9] Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering*, *9*(3), 90-95. <https://doi.org/10.1109/MCSE.2007.55>

[10] Lee, C. C. (1990). Fuzzy logic in control systems: Fuzzy logic controller, Part I. *IEEE Transactions on Systems, Man, and Cybernetics*, *20*(2), 404-418. <https://doi.org/10.1109/21.52551>

[11] Kose, U., & Arslan, A. (2017). Forecasting chaotic time series via ANFIS supported by vortex optimization algorithm: Applications on electroencephalogram time series. *Arabian Journal for Science and Engineering*, *42*(8), 3103-3114. <https://doi.org/10.1007/s13369-016-2279-z>

[12] Ross, T. J. (2017). *Fuzzy Logic with Engineering Applications* (4th ed.). Wiley.

[13] The scikit-fuzzy developers (2026). *scikit-fuzzy: Fuzzy logic toolkit for SciPy [Python package]*. <https://github.com/scikit-fuzzy/scikit-fuzzy>